# DyslexiaLens — Training Prototype (Colab + 28×28)

A standalone CNN that screens handwriting photos for dyslexia indicators.

This notebook walks through the full training pipeline:

1. (Colab only) Unzip the dataset uploaded to `/content/` and remove the zip to free space.
2. Load the labels CSV and remap the on-disk image folder to absolute paths.
3. Build a `tf.data` pipeline (Grayscale PNG decode → Normalize) sized for the **native 28×28** images — *no upscaling*.
4. Define a Functional-API CNN with a **Custom Layer** (`AdaptiveContrastNorm`).
5. Train with a **Custom Loss** (`BinaryFocalLoss`) for class imbalance and a **Custom Callback** (`TrainingMonitor`).
6. Resume from an existing `dyslexia_model.keras` checkpoint if one is present, log to TensorBoard (without histograms — keeps logs small), save best checkpoints, and export the production `.keras` artefact.

All custom components live in `dyslexia_components.py` so the inference service (`inference.py`, `api.py`, `gradcam.py`) can re-load them with the same `custom_objects` registry.

In [ ]:
import datetime
import os
import sys

# ----- Project & dataset names ---------------------------------------------
CSV_NAME  = "Dataset_Dyslexia_NoAugmentation.csv"
DATA_NAME = "Dataset_Gambo_NoAugmentation"

# ----- Environment detection (Colab vs local Jupyter) ----------------------
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# ----- Colab: extract the manually-uploaded dataset zip --------------------
# The user uploads `Dataset_Gambo_NoAugmentation.zip` to /content/. We unzip
# it to /content/local_data, then delete the zip (equivalent to `!rm`) so
# the runtime disk doesn't carry both copies.
def _resolve_data_root(parent: str) -> str | None:
    """Return the directory that actually contains `Train/` and `Test/`.
    Some zips place those folders directly at the root, others nest them
    inside a `Dataset_Gambo_NoAugmentation/` subfolder."""
    if not os.path.isdir(parent):
        return None
    for cand in (parent, os.path.join(parent, DATA_NAME)):
        if os.path.isdir(os.path.join(cand, "Train")) and os.path.isdir(
            os.path.join(cand, "Test")
        ):
            return cand
    return None

if IS_COLAB:
    ZIP_PATH    = f"/content/{DATA_NAME}.zip"
    EXTRACT_DIR = "/content/local_data"

    if _resolve_data_root(EXTRACT_DIR) is None:
        if not os.path.exists(ZIP_PATH):
            raise FileNotFoundError(
                f"Upload {os.path.basename(ZIP_PATH)} to /content/ first."
            )
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        print(f"Unzipping {ZIP_PATH} -> {EXTRACT_DIR} ...")
        import zipfile
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        os.remove(ZIP_PATH)            # same effect as !rm, more portable
        print("Unzip complete. Zip file removed to save disk space.")
    else:
        print(f"Data already extracted under {EXTRACT_DIR}; skipping unzip.")

# ----- Project root discovery (CSV anchor) ---------------------------------
# Anchor every path to the project root so the notebook works no matter
# which directory `jupyter notebook` was launched from. Override manually if
# auto-discovery fails:
#     BASE_DIR = "/content"   # for Colab
#     BASE_DIR = r"D:\path\to\project"   # for Windows
def _find_project_root(marker: str, max_up: int = 6) -> str:
    candidates = []
    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        candidates.append(os.path.dirname(os.path.abspath(vsc)))
    candidates.append(os.path.abspath(os.getcwd()))

    seen = set()
    for start in candidates:
        cur = start
        for _ in range(max_up + 1):
            if cur in seen:
                break
            seen.add(cur)
            if os.path.exists(os.path.join(cur, marker)):
                return cur
            parent = os.path.dirname(cur)
            if parent == cur:
                break
            cur = parent
        if os.path.isdir(start):
            for child in os.listdir(start):
                child_path = os.path.join(start, child)
                if os.path.isdir(child_path) and os.path.exists(
                    os.path.join(child_path, marker)
                ):
                    return child_path
    raise FileNotFoundError(
        f"Could not locate '{marker}'. Searched from: {candidates}. "
        "Set BASE_DIR manually in this cell."
    )

BASE_DIR = _find_project_root(CSV_NAME)
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# ----- Heavy imports (after BASE_DIR is on sys.path) -----------------------
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers

from dyslexia_components import (
    AdaptiveContrastNorm,
    BinaryFocalLoss,
    TrainingMonitor,
    CUSTOM_OBJECTS,
)

# ----- Path setup ----------------------------------------------------------
CSV_PATH        = os.path.join(BASE_DIR, CSV_NAME)
CSV_PATH_PREFIX = os.path.join("Dataset", "Gambo")    # prefix used in CSV

if IS_COLAB:
    DATA_ROOT  = _resolve_data_root(EXTRACT_DIR)        # auto-detects layout
    if DATA_ROOT is None:
        raise FileNotFoundError(
            f"No 'Train' / 'Test' folders found under {EXTRACT_DIR}. "
            "Re-upload the dataset zip and re-run this cell."
        )
    MODEL_PATH = "/content/dyslexia_model.keras"
else:
    DATA_ROOT  = os.path.join(BASE_DIR, DATA_NAME)
    MODEL_PATH = os.path.join(BASE_DIR, "dyslexia_model.keras")

# ----- Hyper-parameters ----------------------------------------------------
IMG_SIZE   = (28, 28)        # native size of the dataset, no upscaling
BATCH_SIZE = 256             # tiny images -> a much larger batch fits easily
EPOCHS     = 12
SEED       = 42

LOG_DIR = os.path.join(BASE_DIR, "logs", "fit",
                       datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TensorFlow :", tf.__version__)
print("Environment:", "Colab" if IS_COLAB else "local")
print("BASE_DIR   :", BASE_DIR)
print("DATA_ROOT  :", DATA_ROOT)
print("MODEL_PATH :", MODEL_PATH)
print("Log dir    :", LOG_DIR)

In [ ]:
df = pd.read_csv(CSV_PATH)


def remap_path(p: str) -> str:
    """CSV stores paths as 'Dataset\\Gambo\\Train\\...'. The actual folder
    on disk is `DATA_ROOT`, so swap the prefix, normalise separators, and
    anchor the result to DATA_ROOT so it works from any CWD."""
    p = p.replace("\\", os.sep).replace("/", os.sep)
    parts = p.split(os.sep)
    if len(parts) >= 2 and os.sep.join(parts[:2]) == CSV_PATH_PREFIX:
        parts = parts[2:]
    return os.path.join(DATA_ROOT, *parts)


df["full_path"] = df["image_path"].apply(remap_path)

# Sanity-check that the first remapped path exists on disk.
sample_path = df["full_path"].iloc[0]
assert os.path.exists(sample_path), f"Path remap failed: {sample_path}"

train_df = df[df["split"] == "Train"].reset_index(drop=True)
val_df   = df[df["split"] == "Validation"].reset_index(drop=True)
test_df  = df[df["split"] == "Test"].reset_index(drop=True)

print(f"Train     : {len(train_df):>7}")
print(f"Validation: {len(val_df):>7}")
print(f"Test      : {len(test_df):>7}")
print("\nTrain class balance:")
print(train_df["target_class"].value_counts().sort_index())
df.head(3)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def _decode_and_normalize(path):
    """Read a small grayscale PNG and normalise to [0, 1].
    Most images are already 28x28, but a few are off-by-one (e.g. 29x29),
    so we resize down to IMG_SIZE to guarantee a uniform batch shape.
    No upscaling — IMG_SIZE stays at the native 28x28 resolution."""
    raw = tf.io.read_file(path)
    img = tf.image.decode_png(raw, channels=1)
    img = tf.image.resize(img, IMG_SIZE, method="area")
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape((IMG_SIZE[0], IMG_SIZE[1], 1))
    return img


def _augment(img):
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    return tf.clip_by_value(img, 0.0, 1.0)


def _make_ds(frame, training: bool):
    paths = frame["full_path"].values
    labels = frame["target_class"].astype("float32").values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=8192, seed=SEED,
                        reshuffle_each_iteration=True)

    def _load(path, label):
        img = _decode_and_normalize(path)
        # Augmentation disabled for the baseline run: at 28x28 even small
        # brightness/contrast shifts can erase weak handwriting features.
        # Re-enable once the model is reliably above the random baseline.
        # if training:
        #     img = _augment(img)
        return img, label

    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = _make_ds(train_df, training=True)
val_ds   = _make_ds(val_df,   training=False)
test_ds  = _make_ds(test_df,  training=False)

# Sanity check on one batch.
for imgs, lbls in train_ds.take(1):
    print("Batch:", imgs.shape, lbls.shape,
          "min/max:", float(tf.reduce_min(imgs)),
          float(tf.reduce_max(imgs)))

## Custom Components

Three custom Keras objects are defined in `dyslexia_components.py`:

| Component | Type | Purpose |
|---|---|---|
| `AdaptiveContrastNorm` | Custom Layer | Per-image spatial contrast normalisation with a learnable affine — robust to phone-camera lighting variance. |
| `BinaryFocalLoss` | Custom Loss | Down-weights easy / over-represented samples (Train is 71% positive). |
| `TrainingMonitor` | Custom Callback | Streams per-epoch metrics to a JSON history file for downstream dashboards. |

`AdaptiveContrastNorm` and `BinaryFocalLoss` are registered with `@keras.utils.register_keras_serializable`, so the saved `.keras` artefact loads cleanly inside `inference.py` and during the resume-from-checkpoint logic below.

In [ ]:
# Diagnostic flag: set False to bypass AdaptiveContrastNorm and see if the
# bare CNN can learn from the raw pixels. Flip back to True once the
# 'naked' pipeline is confirmed to converge above the random baseline.
USE_CUSTOM_LAYER = False

def build_dyslexia_model() -> tf.keras.Model:
    """Compact CNN sized for native 28x28 grayscale handwriting tiles.

    Two max-pools take the spatial map 28 -> 14 -> 7. The third conv block
    (named `last_conv`) keeps a 7x7 feature map so Grad-CAM has enough
    spatial resolution to produce a meaningful heatmap.
    """
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 1), name="input_layer")

    # Per-image contrast normalisation (Custom Layer).
    if USE_CUSTOM_LAYER:
        x = AdaptiveContrastNorm(name="adaptive_contrast")(inputs)
    else:
        x = inputs            # diagnostic: raw normalised pixels straight in

    # Block 1: 28 -> 14
    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1_a")(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1_b")(x)
    x = layers.MaxPool2D(2, name="pool1")(x)

    # Block 2: 14 -> 7
    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2_a")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2_b")(x)
    x = layers.MaxPool2D(2, name="pool2")(x)

    # Block 3: 7 -> 7  (last_conv is the layer Grad-CAM hooks into)
    x = layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3_a")(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu", name="last_conv")(x)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu", name="dense_1")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="dyslexia_prob")(x)

    return tf.keras.Model(inputs, outputs, name="DyslexiaLens_v1")


model = build_dyslexia_model()
model.summary()

## Visual sanity check

Plot 5 samples *after* they pass through `_decode_and_normalize` so we can confirm:

* the images aren't solid black/white blocks,
* the [0, 1] normalisation hasn't crushed the strokes,
* the labels look sensible against the picture (handwriting that *looks* dyslexic should usually carry `target_class=1`).

If these look off, the issue is in the data pipeline — not the loss, not the model.

In [ ]:
import collections
import matplotlib.pyplot as plt

# Pull 5 random samples (the train pipeline shuffles, so .take is randomised).
samples = list(train_ds.unbatch().take(5))

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, (img, lbl) in zip(axes, samples):
    arr = img.numpy().squeeze()
    ax.imshow(arr, cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(f"label={int(lbl)}  range=[{arr.min():.2f}, {arr.max():.2f}]")
    ax.axis("off")
fig.suptitle("Samples after _decode_and_normalize")
plt.tight_layout()
plt.show()

# Wider statistical snapshot to detect silent failures (e.g. all-black images).
wide = list(train_ds.unbatch().take(1024))
lbls = [int(y.numpy()) for _, y in wide]
means = [float(x.numpy().mean()) for x, _ in wide]
print("Class balance in 1024 samples :", collections.Counter(lbls))
print(f"Pixel mean (mean of means)   : {sum(means)/len(means):.4f}")
print(f"Pixel mean (min / max sample): {min(means):.4f} / {max(means):.4f}")
print(f"Per-image dynamic range mean : "
      f"{sum(float(x.numpy().max()-x.numpy().min()) for x,_ in wide)/len(wide):.4f}")

In [ ]:
classes = np.array([0, 1])
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["target_class"].values,
)
class_weight = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
print("Class weights:", class_weight)

# ----- Reset stuck checkpoint before starting a fresh run ------------------
# Set RESET_CHECKPOINT = False on subsequent runs to resume training
# from the best-so-far checkpoint instead of starting over.
RESET_CHECKPOINT = True
if RESET_CHECKPOINT and os.path.exists(MODEL_PATH):
    os.remove(MODEL_PATH)
    print(f"Removed stale checkpoint: {MODEL_PATH}")

# ----- Resume from an existing checkpoint if one is present ----------------
# `.keras` saves the optimizer state, so loading a checkpoint resumes
# training exactly where it left off. If no file exists we compile the
# freshly-built `model` from the previous cell.
if os.path.exists(MODEL_PATH):
    print(f"Resuming training from existing checkpoint: {MODEL_PATH}")
    model = tf.keras.models.load_model(MODEL_PATH, custom_objects=CUSTOM_OBJECTS)
    print(f"   Loaded model has {model.count_params():,} parameters.")
else:
    print(f"No checkpoint at {MODEL_PATH} - compiling a fresh model.")
    # HARD-RESET DIAGNOSTIC: plain BinaryCrossentropy + no class weighting.
    # We want to confirm the bare CNN can at least collapse to the majority
    # class (~71% accuracy on this dataset). If even this can't converge,
    # the issue is the data pipeline / label mapping, not the loss.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

cb_list = [
    # histogram_freq=0 keeps TensorBoard logs small (no per-weight histograms).
    tf.keras.callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=0),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1,
    ),
    TrainingMonitor(history_path=os.path.join(LOG_DIR, "history.json")),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    # class_weight removed for the hard-reset diagnostic.
    callbacks=cb_list,
    verbose=1,
)

In [ ]:
print("\n=== Test-set evaluation ===")
results = model.evaluate(test_ds, return_dict=True, verbose=1)
for k, v in results.items():
    print(f"{k:>10}: {v:.4f}")

print("\nTargets : accuracy > 0.85, mae < 0.02")
print("Accuracy:", "HIT" if results["accuracy"] > 0.85 else "MISS",
      f"({results['accuracy']:.4f})")
print("MAE     :", "HIT" if results["mae"]      < 0.02 else "MISS",
      f"({results['mae']:.4f})")

# Persist the final model (overwrites the best-so-far checkpoint with the
# restored-best-weights state from EarlyStopping).
model.save(MODEL_PATH)
print(f"\nModel saved to {MODEL_PATH}")
print(f"TensorBoard: tensorboard --logdir {os.path.dirname(LOG_DIR)}")